<a href="https://colab.research.google.com/github/vishal6975/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishal6975/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. What one row means: one content page, for one client, on one specific day
   (from fact_content_daily_performance).
2. Table(s) I'll use: dim_content (page metadata) joined with
   fact_content_daily_performance (daily metrics).
3. Time window: a mid-panel month, month=2026-03, for feature development.
4. What I'd predict/rank: a proxy label for "needs review" — e.g. whether a
   page's impressions or clicks declined over the month.
5. What I deliberately exclude: any FlyRank product-computed scores like
   health_score or priority_score — these are decision outputs, not raw signals,
   and using them would leak the answer into my model.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Feature fields

I will use observed search and traffic signals from the daily performance table:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`
- `ga4_users`
- `ga4_engaged_sessions`
- `ga4_total_engagement_sec`
- `sessions_organic`
- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`
- `sessions_chatgpt`
- `scroll_events`

These are measured performance signals. I will only use fields that are available before the decision point.

### Label / proxy field

- `declined_this_month` — a derived directional label based on comparing early-March and late-March `gsc_impressions`.

This is a proxy for "needs review". It does not mean that the page definitely needs a specific action.

### Context fields

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`
- `month`

These fields identify the observation, provide time information, or describe data availability. They are used for joins, grouping, filtering, and quality checks rather than as model inputs.

### Derived feature

- `gsc_ctr` — calculated as `gsc_clicks / gsc_impressions` when impressions are greater than zero.

I derive this rather than referring to a non-existent `ctr` column.

### Excluded fields

I exclude any FlyRank product-computed decision fields such as `health_score`, `priority_score`, `action_type`, and `refresh_tier` if they are present in other joined tables.

These are decision outputs rather than raw evidence. Including them as model features could cause leakage and allow the model to reproduce an existing product rule instead of learning from underlying signals.

I also exclude the target `declined_this_month` from the feature set because it is the label being predicted.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# Install required packages
!pip -q install -U duckdb huggingface_hub

import duckdb
from huggingface_hub import login, get_token

# ---------------------------------------------------------
# 1. Login to Hugging Face
# ---------------------------------------------------------

login()

# ---------------------------------------------------------
# 2. Get the token from the Hugging Face login
# ---------------------------------------------------------

token = get_token()

if not token:
    raise RuntimeError(
        "Hugging Face login failed. Please run the login again "
        "and make sure you use an account with access to the FlyRank dataset."
    )

# ---------------------------------------------------------
# 3. Connect DuckDB
# ---------------------------------------------------------

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# ---------------------------------------------------------
# 4. Create Hugging Face authentication secret
# ---------------------------------------------------------

con.execute("DROP SECRET IF EXISTS hf_token;")

con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{token}'
    );
""")

print("✅ Hugging Face authentication successful!")
print("✅ DuckDB is ready to access the FlyRank dataset.")

✅ Hugging Face authentication successful!
✅ DuckDB is ready to access the FlyRank dataset.


In [6]:
test = con.execute("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    LIMIT 5
""").df()

print("✅ Dataset access successful!")
print(test)

✅ Dataset access successful!
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           

In [8]:
# Show the actual columns available in the March 2026 dataset

columns = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

print(columns[["column_name", "column_type"]].to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

In [10]:
# =========================================================
# SECTION 3 — VERIFY THE DATA CONTRACT
# =========================================================

DATA_PATH = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# ---------------------------------------------------------
# 0. Get the actual schema from the data
# ---------------------------------------------------------

schema = con.execute(f"""
    DESCRIBE SELECT * FROM {DATA_PATH}
""").df()

available_columns = set(schema["column_name"].tolist())

print("Actual columns detected:", len(available_columns))


# ---------------------------------------------------------
# 1. Grain check
# One row = one content page + one client + one date
# ---------------------------------------------------------

grain_check = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        COUNT(*) AS cnt
    FROM {DATA_PATH}
    GROUP BY
        content_hash_id,
        client_hash_id,
        report_date
    HAVING COUNT(*) > 1
""").df()

print("\n1. Grain check")
print("Duplicate grain rows (should be 0):", len(grain_check))


# ---------------------------------------------------------
# 2. Row count and date span
# ---------------------------------------------------------

counts = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {DATA_PATH}
""").df()

print("\n2. Row count and date span:")
print(counts.to_string(index=False))


# ---------------------------------------------------------
# 3. GA4 availability
# ---------------------------------------------------------

avail = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN 1
                ELSE 0
            END
        ) AS ga4_true_rows,

        ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS ga4_available_pct

    FROM {DATA_PATH}
""").df()

print("\n3. GA4 availability:")
print(avail.to_string(index=False))


# ---------------------------------------------------------
# 4. Missing-value check
# Only check columns that ACTUALLY exist.
# ---------------------------------------------------------

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

# Keep only fields that really exist
checked_columns = [
    col for col in feature_columns
    if col in available_columns
]

missing_expressions = []

for col in checked_columns:
    missing_expressions.append(
        f"""
        SUM(
            CASE
                WHEN "{col}" IS NULL
                THEN 1
                ELSE 0
            END
        ) AS missing_{col}
        """
    )

missing_query = f"""
    SELECT
        COUNT(*) AS total_rows,
        {",".join(missing_expressions)}
    FROM {DATA_PATH}
"""

missing = con.execute(missing_query).df()

print("\n4. Missing values:")
print(missing.to_string(index=False))

print("\nFields checked for missing values:")
print(", ".join(checked_columns))


# ---------------------------------------------------------
# 5. Derived CTR check
# CTR = clicks / impressions
# Only calculate when impressions > 0.
# ---------------------------------------------------------

if "gsc_impressions" in available_columns and "gsc_clicks" in available_columns:

    ctr_check = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,

            SUM(
                CASE
                    WHEN gsc_impressions > 0
                     AND gsc_clicks IS NOT NULL
                    THEN 1
                    ELSE 0
                END
            ) AS rows_with_calculable_ctr,

            ROUND(
                100.0 *
                SUM(
                    CASE
                        WHEN gsc_impressions > 0
                         AND gsc_clicks IS NOT NULL
                        THEN 1
                        ELSE 0
                    END
                ) / COUNT(*),
                2
            ) AS calculable_ctr_pct

        FROM {DATA_PATH}
    """).df()

    print("\n5. Derived CTR check:")
    print(ctr_check.to_string(index=False))

else:
    print("\n5. Derived CTR check skipped:")
    print("gsc_impressions or gsc_clicks is not available.")


# ---------------------------------------------------------
# 6. Daily coverage
# ---------------------------------------------------------

daily_counts = con.execute(f"""
    SELECT
        report_date,
        COUNT(*) AS row_count
    FROM {DATA_PATH}
    GROUP BY report_date
    ORDER BY report_date
""").df()

print("\n6. Daily row counts:")
print(daily_counts.to_string(index=False))


# ---------------------------------------------------------
# 7. Final verification
# ---------------------------------------------------------

print("\n=========================================================")
print("SECTION 3 VERIFICATION COMPLETE")
print("=========================================================")

print("Actual schema columns:", len(available_columns))
print("Feature fields checked:", len(checked_columns))
print("Duplicate grain rows:", len(grain_check))
print("March date range: 2026-03-01 to 2026-03-31")
print("Total March rows: 9,841,378")
print("GA4 available rows: 413,966")
print("GA4 availability: 4.21%")

Actual columns detected: 31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


1. Grain check
Duplicate grain rows (should be 0): 0

2. Row count and date span:
 row_count   min_date   max_date
   9841378 2026-03-01 2026-03-31

3. GA4 availability:
 total_rows  ga4_true_rows  ga4_available_pct
    9841378       413966.0               4.21


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


4. Missing values:
 total_rows  missing_gsc_impressions  missing_gsc_clicks  missing_gsc_sum_position  missing_gsc_avg_position  missing_ga4_pageviews  missing_ga4_sessions  missing_ga4_users  missing_ga4_engaged_sessions  missing_ga4_total_engagement_sec  missing_sessions_organic  missing_sessions_direct  missing_sessions_referral  missing_sessions_social  missing_sessions_paid  missing_sessions_ai  missing_scroll_events
    9841378                      0.0                 0.0                       0.0                 6230317.0              3018741.0             3018741.0          3018741.0                     3018741.0                         3018741.0                 3018741.0                3018741.0                  3018741.0                3018741.0              3018741.0            3018741.0              3018741.0

Fields checked for missing values:
gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4

I verify three claims from my contract: (1) the grain — one row is truly one
page/client/day, (2) the row count and date span of my March 2026 slice, and
(3) how many rows have real GA4 availability using IS TRUE.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This data can support directional decision-support, but it has important limitations.

- **Unbalanced history:** Different content pages may have different numbers of observations. More or fewer observations should not automatically be interpreted as better or worse performance.

- **GSC-only or GA4-unavailable rows:** The March 2026 data contains 9,841,378 rows, while only 413,966 rows (4.21%) have `ga4_data_available IS TRUE`. Therefore, GA4-based signals are not available for every observation.

- **Missing values:** Some performance fields may be unavailable for some rows. Missing data should be measured and treated as missing information rather than automatically converted to zero.

- **Window limitation:** This contract uses March 2026 as the development window. A single month does not represent the complete historical performance of a page.

- **Window overlap:** A page's performance before March or after March may affect how its March trend should be interpreted.

- **Directional label:** A decline in impressions or clicks is only a proxy for "needs review." It does not prove that a page requires a particular action.

- **No causal conclusion:** The available data can show measured changes and associations, but it cannot prove that a specific content change caused a performance change.

- **Generalization limitation:** Patterns observed in March 2026 may not generalize to other months, clients, industries, or search conditions.

- **Decision-output leakage:** Product-computed fields such as `health_score`, `priority_score`, `action_type`, and `refresh_tier` are excluded because they represent existing decisions. Using them as model inputs could create leakage and allow the model to reproduce an existing rule.

- **Decision-support scope:** The data contract can support ranking or flagging pages for review. It should not be interpreted as an autonomous decision to change, remove, or rewrite content.

In [11]:
print("Data limits documented based on the verified March 2026 data.")

Data limits documented based on the verified March 2026 data.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.